<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# BLOQUE A — Integración y Preprocesamiento de Datos

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import leer_imagenes_idx, leer_etiquetas_idx, normalizar_y_guardar, descargar_datos

ModuleNotFoundError: No module named 'src'

## 1. Descarga y Carga de Datos Raw

In [ ]:
# IDs de los archivos en Google Drive
file_ids = {
    'train_X': '1enziBIpqiv_t95KQcifsclNH2BdR8lAd',
    'test_X': '1Jeax6tnQ6Nmr2PTNXdQqzKnN0YqtrLe4',
    'train_Y': '1MZtn2iA5cgiYT1i3O0ECuR01oD0kGHh7',
    'test_Y': '1K5pxwk2s3RDYsYuwv8RftJTXZ-RGR7K4'
}

# Directorio para los datos raw
raw_dir = 'data/raw/'
os.makedirs(raw_dir, exist_ok=True)

# Descargar los archivos
ruta_train_x = descargar_datos(file_ids['train_X'], os.path.join(raw_dir, 'train-images-idx3-ubyte'))
ruta_test_x = descargar_datos(file_ids['test_X'], os.path.join(raw_dir, 't10k-images-idx3-ubyte'))
ruta_train_y = descargar_datos(file_ids['train_Y'], os.path.join(raw_dir, 'train-labels-idx1-ubyte'))
ruta_test_y = descargar_datos(file_ids['test_Y'], os.path.join(raw_dir, 't10k-labels-idx1-ubyte'))

# Cargar los datos raw
X_train_raw = leer_imagenes_idx(ruta_train_x)
y_train_raw = leer_etiquetas_idx(ruta_train_y)
X_test_raw = leer_imagenes_idx(ruta_test_x)
y_test_raw = leer_etiquetas_idx(ruta_test_y)

print(f"Dimensiones de X_train_raw: {X_train_raw.shape}")
print(f"Dimensiones de y_train_raw: {y_train_raw.shape}")
print(f"Dimensiones de X_test_raw: {X_test_raw.shape}")
print(f"Dimensiones de y_test_raw: {y_test_raw.shape}")

## 2. Preprocesamiento y Guardado

In [ ]:
# Normalizar y guardar los datos procesados
normalizar_y_guardar(X_train_raw, X_test_raw, y_train_raw, y_test_raw, out_dir="data/processed/")

## 3. EDA Mínimo

In [ ]:
# Cargar los datos procesados para el EDA
X_train = np.load('data/processed/X_train.npy')
y_train = pd.read_csv('data/processed/y_train.csv')

print("Value counts de las etiquetas de entrenamiento:")
print(y_train['label'].value_counts())

# Visualizar algunas imágenes
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_raw[i], cmap='gray')
    ax.set_title(f"Clase: {y_train['label'][i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# BLOQUE B — Reducción de Dimensionalidad y Visualización

In [ ]:
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import Isomap, TSNE
from sklearn.manifold import SpectralEmbedding
import time

# Cargar los datos procesados
X_train = np.load('data/processed/X_train.npy').reshape(X_train.shape[0], -1)
X_test = np.load('data/processed/X_test.npy').reshape(X_test_raw.shape[0], -1)
y_train = pd.read_csv('data/processed/y_train.csv')

## 1. Implementación de Reductores

In [ ]:
def fit_reducer(X, method, n_components, seed=42, **kwargs):
    start_time = time.time()
    if method == 'pca':
        reducer = PCA(n_components=n_components, random_state=seed)
    elif method == 'nmf':
        reducer = NMF(n_components=n_components, random_state=seed, max_iter=500)
    elif method == 'isomap':
        reducer = Isomap(n_components=n_components)
    elif method == 'spectral':
        reducer = SpectralEmbedding(n_components=n_components, random_state=seed)
    else:
        raise ValueError(f"Método '{method}' no soportado.")
    
    embeddings = reducer.fit_transform(X)
    end_time = time.time()
    
    metadata = {
        'method': method,
        'n_components': n_components,
        'fit_time': end_time - start_time,
        'params': reducer.get_params()
    }
    
    return reducer, embeddings, metadata

def transform_reducer(reducer, X):
    return reducer.transform(X)

def save_embedding(path, arr, meta):
    np.save(path, arr)
    pd.DataFrame([meta]).to_json(path.replace('.npy', '.json'))

## 2. Generación y Guardado de Embeddings

In [ ]:
reductores = ['pca', 'nmf', 'isomap', 'spectral']
n_dims = [10, 50]
embeddings_dir = 'data/processed/embeddings/'
os.makedirs(embeddings_dir, exist_ok=True)

for method in reductores:
    for n in n_dims:
        print(f"Procesando {method} con n_components={n}...")
        reducer, X_train_emb, meta = fit_reducer(X_train, method, n)
        
        # Guardar embedding de entrenamiento
        save_embedding(f"{embeddings_dir}{method}_{n}_train.npy", X_train_emb, meta)
        
        # Transformar y guardar embedding de prueba (si es posible)
        if hasattr(reducer, 'transform'):
            X_test_emb = transform_reducer(reducer, X_test)
            save_embedding(f"{embeddings_dir}{method}_{n}_test.npy", X_test_emb, meta)

## 3. Visualización 2D

In [ ]:
reductores_viz = ['pca', 'nmf', 'isomap', 'spectral']
X_train_sample = X_train[:5000] # Submuestra para visualización
y_train_sample = y_train[:5000]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Visualización 2D de Embeddings', fontsize=16)

for ax, method in zip(axes.flat, reductores_viz):
    _, emb, _ = fit_reducer(X_train_sample, method, 2)
    scatter = ax.scatter(emb[:, 0], emb[:, 1], c=y_train_sample['label'], cmap='viridis', s=10, alpha=0.7)
    ax.set_title(method.upper())
    ax.set_xlabel('Componente 1')
    ax.set_ylabel('Componente 2')

# Añadir t-SNE por separado ya que no tiene 'transform'
print("Procesando t-SNE...")
tsne = TSNE(n_components=2, random_state=42)
X_train_tsne = tsne.fit_transform(X_train_sample)

ax = axes.flat[-1] # Usar el último subplot para t-SNE
scatter = ax.scatter(X_train_tsne[:, 0], X_train_tsne[:, 1], c=y_train_sample['label'], cmap='viridis', s=10, alpha=0.7)
ax.set_title('T-SNE')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('outputs/figures/embeddings_2d.png')
plt.savefig('outputs/figures/embeddings_2d.svg')
plt.show()